# 多智能体：实时观看一个课程团队工作

把任务委派给子智能体的协调器存在可观测性缺口。会话级流会在模型生成时预览主线程的文本，但子智能体的输出要等它整个回合被缓冲后才可见。如果研究员运行两分钟，你就两分钟什么都看不到。

本笔记本通过组合使用 Managed Agents API 的四项功能弥合这一缺口：

- **按线程的增量流式输出。** 每个会话线程的流都接受与会话级流相同的 `event_deltas` 参数，因此子智能体的文本会在它自己的流上实时预览。
- **创建会话时的初始事件。** `sessions.create` 接受 `initial_events`，因此会话在创建它的那次调用中就开始工作。
- **智能体模型上的强度。** `model.effort` 设置 Claude 在每次推理调用上投入多少力气，按智能体分别设置。在团队中，这就是按角色的成本杠杆。
- **更新智能体时可选的版本号。** `agents.update` 把当前版本视为可选的并发键，而不是必需项。

你要构建的团队在为七年级科学的一周单元做规划：协调器把任务委派给课标研究员（web 搜索，高强度）和教案编写者（无网络访问），然后汇总成单元计划。如果 `multiagent` 协调器模式对你还是新鲜的，请先看 [`CMA_coordinate_specialist_team.ipynb`](CMA_coordinate_specialist_team.ipynb)。本笔记本在这些形态之上继续构建。

## 1. 配置客户端

这些功能依赖标准的 `managed-agents-2026-04-01` beta 请求头。本笔记本中的 SDK 调用（`sessions.create` 上的 `initial_events`、线程流上的 `event_deltas`、`model.effort`）需要 `anthropic>=0.118.0`。

In [ ]:
%%capture
%pip install -qU "anthropic>=0.118.0" python-dotenv

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-5")
client = anthropic.Anthropic()

## 2. 创建两个专家智能体

研究员获得 web 搜索和 `high` 强度等级，因为课标对齐是大量依赖判断的工作：它必须权衡哪些能力标准真正契合该单元，而不是把查到的内容全部罗列。`effort` 放在 `model` 对象内，接受 `low`、`medium`、`high`、`xhigh` 或 `max`（可以是裸字符串，也可以是 `{"type": "high"}`）。并非每个模型都接受每个等级，无效组合会在创建时被拒绝。

强度也是团队的主要成本杠杆：较高的等级让 Claude 在每次推理调用上花更多词元，较低的等级则限制这部分开销。因为它是按智能体设置的，你只为需要深度的角色买单。在更大的团队里，排版或分诊类角色可以降到 `low`，而不占用研究员的预算。

在智能体上设置强度，而不是按会话设置：放在按会话的 `model` 覆盖里的 `effort` 等级不会生效。

web 工具的配置还接受允许域名和屏蔽域名列表，因此生产部署可以把这位研究员限定在它应当信任的课标站点。配置形态见[配置工具集](https://platform.claude.com/docs/en/managed-agents/tools#configuring-the-toolset)。

In [ ]:
RESEARCHER_SYSTEM = """You research US middle school science standards.
Given a unit topic and grade level, use web search to find:
- The two or three NGSS performance expectations the unit should target
- What students are expected to already know, and where the topic leads next
- Two or three documented student misconceptions for the topic
Return via send_to_parent:
{"standards": [...], "prior_knowledge": ..., "leads_to": ...,
 "misconceptions": [...], "sources": [...]}"""

standards_researcher = client.beta.agents.create(
    name="standards_researcher",
    description="Finds the NGSS standards and documented misconceptions for a science topic.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL, "effort": "high"},
    system=RESEARCHER_SYSTEM,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [{"name": "web_search"}, {"name": "web_fetch"}],
        }
    ],
    betas=BETAS,
)

教案编写者只依据协调器交给它的材料工作。禁用 web 工具能让它的教案立足于研究员核实过的发现，而不是某次新搜索返回的任何内容。

In [ ]:
WRITER_SYSTEM = """You write middle school lesson plans.
You will be given a topic, grade level, target standards, and known misconceptions.
Draft five 45-minute lessons, one per day. For each day give:
- Objective (tied to a target standard)
- Warm-up (5 min)
- Main activity (30 min)
- Exit ticket (one question that probes a misconception where possible)
Return via send_to_parent: {"days": [{"day": 1, "objective": ..., "warm_up": ...,
"main_activity": ..., "exit_ticket": ...}, ...]}"""

lesson_writer = client.beta.agents.create(
    name="lesson_writer",
    description="Drafts a day-by-day lesson sequence from standards and misconceptions.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL},
    system=WRITER_SYSTEM,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
    betas=BETAS,
)

创建响应会回显已解析的模型配置，包括你省略的字段。研究员显示你设置的 `high`，编写者显示该模型的默认强度。如果 `effort` 返回 `None`，说明你所在组织的 beta 请求头尚未支持该功能：该字段被丢弃而非拒绝，所以这个回显是发现问题的地方。

In [ ]:
for agent in (standards_researcher, lesson_writer):
    effort = getattr(agent.model, "effort", None)
    print(f"{agent.name}: {agent.id} v{agent.version} effort={effort and effort.type}")

## 3. 创建协调器

协调器承载 `multiagent` 名册。派生和委派工具由名册自动注入，并且名册把每个子智能体锁定在当前版本上。这一锁定在你之后更新研究员时很重要。

In [ ]:
COORDINATOR_SYSTEM = """You plan one-week teaching units.
Given a topic and grade level:
1. Send the topic and grade to standards_researcher.
2. When the researcher reports back, send the topic, grade, target standards, and
misconceptions to lesson_writer.
3. Write /mnt/session/outputs/unit_plan.md with sections: Overview, Standards alignment,
Day-by-day plan (from the writer), Misconceptions to watch for, Materials.
Keep it under three pages. To revise the plan, rewrite the whole file with the write
tool rather than patching it with edit. While a subagent is working, wait silently: no
status commentary until its report arrives."""

coordinator = client.beta.agents.create(
    name="unit_planner",
    description="Plans one-week teaching units by delegating research and drafting.",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-watch-subagents-live"},
    model={"id": MODEL},
    system=COORDINATOR_SYSTEM,
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [standards_researcher.id, lesson_writer.id],
    },
    betas=BETAS,
)
print(f"{coordinator.name}: {coordinator.id} v{coordinator.version}")

## 4. 在创建会话时内联写入第一条消息

不使用 `initial_events` 时，启动会话需要两次调用：先创建，再发送一条 `user.message`。在创建时预置第一条消息，就把这两次调用合并成一次。该数组最多接受 50 个 `user.message` 和 `user.define_outcome` 事件，按顺序处理，且校验是全有或全无：任一事件失败，就不会创建会话。非空列表会在同一次调用中启动智能体循环，因此会话返回时已经在向 `running` 推进。

如果你想让单元计划按评分细则打分，可在这里加入一条 `user.define_outcome` 事件（它必须包含 `rubric`，且每次创建只允许一条）。[`CMA_verify_with_outcome_grader.ipynb`](CMA_verify_with_outcome_grader.ipynb) 讲了那个循环。

In [ ]:
import time

env = client.beta.environments.create(
    name="curriculum-studio",
    config={"type": "anthropic_cloud", "networking": {"type": "unrestricted"}},
    betas=BETAS,
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=env.id,
    title="Unit plan: water cycle, grade 7",
    initial_events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": "Plan a one-week unit on the water cycle for 7th grade science. "
                    "Classes are 45 minutes. Write the plan to /mnt/session/outputs/unit_plan.md.",
                }
            ],
        }
    ],
    betas=BETAS,
)

deadline = time.monotonic() + 60
while session.status == "idle" and time.monotonic() < deadline:
    time.sleep(1)
    session = client.beta.sessions.retrieve(session.id, betas=BETAS)

print(f"{session.id}: {session.status}")
if session.status == "idle":
    events = client.beta.sessions.events.list(session.id, limit=1, betas=BETAS)
    if next(iter(events), None) is None:
        raise RuntimeError(
            "The session has no events: your organization's beta header does not carry "
            "initial_events yet. The field is dropped at the gateway, not rejected."
        )
    raise RuntimeError(
        "The seeded message is in the event log but the session is still idle after 60s. "
        "Re-run this cell to keep waiting."
    )

## 5. 实时观看整个团队

预览在设计上就是按线程划分的。一个连接只预览它所读取的线程：会话级流预览主线程，而某个子智能体的预览只出现在该子智能体自己的流上，路径为 `GET /v1/sessions/{session_id}/threads/{thread_id}/stream`。因此模式是每个线程一条流：为协调器读取会话级流，每当 `session.thread_created` 事件宣布一个子智能体（它带有 `session_thread_id` 和 `agent_name`），就在工作线程里为该线程的流挂上一个观察者。

两条规则塑造了这段代码：

- **预览是草稿缓冲区，缓冲事件才是正式记录。** 增量是尽力而为的，在负载下可能停止。随后到达的缓冲 `agent.message` 携带完整内容。SDK 的 `accumulate_managed_agents_event` 辅助函数把 `event_start`、`event_delta` 和缓冲事件合并成一份快照，并在记录到达时替换预览。每个流连接运行一个累加器。
- **没有重放。** 在某个模型请求开始之后才打开的连接不会收到该进行中的请求的任何增量，重连也从不重放错过的增量。在 `session.thread_created` 到达时立刻挂上观察者，就能从子智能体的第一个响应起捕捉它的工作。

每个观察者在 `session.thread_status_idle` 上退出——这是子智能体回合结束时发出的事件。

In [ ]:
import threading

from anthropic.lib.sessions import accumulate_managed_agents_event

print_lock = threading.Lock()
current_speaker = None
reconciliations = []


def say(speaker, text):
    """Print streamed text, labeling it whenever the speaker changes."""
    global current_speaker
    with print_lock:
        if speaker != current_speaker:
            print(f"\n\n=== {speaker} ===")
            current_speaker = speaker
        print(text, end="", flush=True)


def text_of(event):
    if event is None:
        return ""
    return "".join(block.text for block in event.content if block.type == "text")


def fold_preview(snapshot, ev, speaker):
    """Fold one preview event into the speaker's snapshot and return the new snapshot.

    Prints delta text as it arrives. When the buffered record lands, logs a
    reconciliation row comparing it against the accumulated preview. The id
    check guards the case where every delta for a message was shed: the
    snapshot then still holds the previous message, not a preview of this one.
    """
    if ev.type == "event_delta":
        say(speaker, ev.delta.content.text)
    elif ev.type == "agent.message":
        preview = text_of(snapshot) if snapshot and snapshot.id == ev.id else ""
        final = text_of(ev)
        reconciliations.append((speaker, len(preview), len(final), final.startswith(preview)))
    return accumulate_managed_agents_event(snapshot, ev)


def watch_child(thread_id, agent_name):
    """Stream one child thread, previewing its text as the model generates it."""
    snapshot = None
    with client.beta.sessions.threads.events.stream(
        thread_id,
        session_id=session.id,
        event_deltas=["agent.message"],
        betas=BETAS,
    ) as thread_stream:
        for ev in thread_stream:
            if ev.type in ("event_start", "event_delta", "agent.message"):
                snapshot = fold_preview(snapshot, ev, agent_name)
            elif ev.type == "agent.tool_use":
                detail = ev.input.get("query", "") if ev.name == "web_search" else ""
                say(agent_name, f"\n[{ev.name}] {detail}")
            elif ev.type in ("session.thread_status_idle", "session.thread_status_terminated"):
                break

主循环对协调器的文本使用同一个 `fold_preview`，另外处理只在主线程上出现的协调类事件：子智能体派生时的 `session.thread_created`，协调器移交任务时的 `agent.thread_message_sent`，以及子智能体回报时的 `agent.thread_message_received`。从子线程交叉发布的工具使用带有 `session_thread_id`，所以循环会跳过这些：子智能体自己的观察者会显示它们。

循环在遇到任意 `session.status_idle` 时结束，并在停止原因不是 `end_turn` 时打印该原因：等待工具确认或重试耗尽的会话应当结束该单元格，而不是让它挂住。两个循环也会在终止状态事件上退出，因此不可恢复的会话错误会结束单元格，而不是让流保持打开。

In [ ]:
watchers = []
snapshot = None

with client.beta.sessions.events.stream(
    session.id, event_deltas=["agent.message"], betas=BETAS
) as stream:
    for ev in stream:
        if ev.type in ("event_start", "event_delta", "agent.message"):
            snapshot = fold_preview(snapshot, ev, "unit_planner")
        elif ev.type == "session.thread_created":
            say("unit_planner", f"\n[spawned {ev.agent_name}]")
            watcher = threading.Thread(
                target=watch_child,
                args=(ev.session_thread_id, ev.agent_name),
                daemon=True,
            )
            watcher.start()
            watchers.append(watcher)
        elif ev.type == "agent.thread_message_sent":
            say("unit_planner", f"\n[task sent to {ev.to_agent_name}]")
        elif ev.type == "agent.thread_message_received":
            say("unit_planner", f"\n[{ev.from_agent_name} reported back]")
        elif ev.type == "agent.tool_use" and not ev.session_thread_id:
            # session_thread_id marks a call cross-posted from a child thread;
            # that child's watcher already shows it
            say("unit_planner", f"\n[{ev.name}]")
        elif ev.type == "session.status_terminated":
            break
        elif ev.type == "session.status_idle":
            if ev.stop_reason.type != "end_turn":
                say("unit_planner", f"\n[idle without end_turn: {ev.stop_reason.type}]")
            break

for watcher in watchers:
    watcher.join(timeout=60)
if any(watcher.is_alive() for watcher in watchers):
    print("\n[a child stream is still open after the join timeout]")

如果流连上了但始终只投递缓冲事件，说明 `event_deltas` 参数被剥离而非拒绝：这就是 beta 请求头不支持该功能的特征。线程流 URL 返回 404 意味着路径错误或缺少 managed-agents beta 请求头（线程端点受 beta 门控）。路径是 `/threads/{thread_id}/stream`，而不是 `/threads/{thread_id}/events/stream`。

完整的流式输出契约（增量形态、对账保证以及排错表）见[事件与流式输出](https://platform.claude.com/docs/en/managed-agents/events-and-streaming#event-deltas)。

## 6. 把预览与记录对照检查

把一条预览的增量拼接起来，得到的是缓冲事件文本的前缀：是前缀，不一定等于全部文本，因为在负载下增量可能被丢弃。正是这一保证让对账只需一次替换。下面每一行是一条缓冲的 `agent.message`，与记录到达时累加器已构建的预览进行对比。

In [ ]:
for agent_name, preview_chars, final_chars, is_prefix in reconciliations:
    print(
        f"{agent_name:22s} preview {preview_chars:5d} chars | "
        f"final {final_chars:5d} chars | preview is prefix: {is_prefix}"
    )

## 7. 读取单元计划

协调器的提示词要求用整文件 `write` 调用撰写计划，所以完整文档在事件日志中，而对文件的最后一次 `write` 就是最终版本。如果你自己的智能体用 `edit` 修补文件，就读取文件本身，而不是重放写入。

In [ ]:
unit_plan = ""
for ev in client.beta.sessions.events.list(session.id, limit=1000, betas=BETAS):
    if (
        ev.type == "agent.tool_use"
        and ev.name == "write"
        and ev.input.get("file_path", "").endswith("unit_plan.md")
    ):
        unit_plan = ev.input["content"]  # keep the last write, in case of revisions

# Show the structure rather than the full plan. Print unit_plan to read it all.
for line in unit_plan.splitlines():
    if line.startswith("#"):
        print(line)

## 8. 不发版本往返也能上线一处提示词改动

假设评审意见要求每一处对齐结论都要在旁边标注课标的编号。`agents.update` 把智能体当前的 `version` 视为可选的并发键：省略它，更新就无条件生效，由服务器替你递增版本号。配置脚本不必为了写回智能体而先读取它。

当可能存在并发写入者时提供 `version`（版本不匹配会返回 409，因此你始终从已知状态出发更新）。当只有一条流程独占该智能体时省略它，例如同步已入库的智能体定义的 CI 任务。

In [ ]:
updated = client.beta.agents.update(
    standards_researcher.id,
    system=RESEARCHER_SYSTEM
    + "\nCite the performance expectation code (for example MS-ESS2-4) next to every "
    "alignment claim.",
    betas=BETAS,
)
print(f"standards_researcher v{standards_researcher.version} -> v{updated.version}")

多智能体配置有一个注意点：协调器的名册在创建协调器时锁定了研究员的版本，所以已有的协调器仍会委派给旧版本。更新协调器（它的 `multiagent` 字段，或任意字段），就能按最新的子智能体版本重新解析名册。

## 9. 清理

In [ ]:
from utilities import wait_for_idle_status

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id, betas=BETAS)
client.beta.environments.archive(env.id, betas=BETAS)
print("archived")